# Module 3 - Power BI semantic dataset: Import and live

![Power BI mockup](../assets/images/powerbi_report_mockup.png)

The model is built in Databricks. Power BI should be a thin reporting layer with
only the measures and interactions that belong in the report.

In [ ]:
%run ../setup/00_setup

## Runtime pre-check

Run Module 2 first. This notebook expects the dashboard fact and monthly
aggregate to exist.

In [ ]:
required_tables = [
    f"{GOLD}.fact_sales_dashboard",
    f"{GOLD}.fact_sales_dashboard_monthly",
]

missing = [table for table in required_tables if not spark.catalog.tableExists(table)]
if missing:
    raise Exception("Missing BI-ready Gold tables. Run Module 2 first: " + ", ".join(missing))

print("[OK] BI-ready Gold tables exist")

## Import vs live/DirectQuery

![Import vs DirectQuery](../assets/images/import_vs_directquery.png)

## Power BI connector context

![Power BI DirectQuery connector](../assets/images/source_powerbi_directquery_connector.webp)

Use this as a visual walkthrough before opening Power BI Desktop. The key point
for analysts: connector mode is a modelling decision, not just a technical
checkbox.

## Connection walkthrough

![Power BI connection walkthrough](../assets/images/powerbi_connection_walkthrough.png)

Trainer notes:

- show where SQL Warehouse Server hostname and HTTP path live,
- explain authentication before discussing tables,
- select only Gold objects,
- pick Import first unless a live use case is explicit.

In [ ]:
spark.sql(f"""
CREATE OR REPLACE VIEW {GOLD}.v_fact_sales_incremental AS
SELECT
  order_date,
  order_id,
  line_id,
  customer_id,
  segment,
  customer_region,
  product_id,
  category,
  subcategory,
  channel,
  status,
  quantity,
  line_revenue,
  line_margin
FROM {GOLD}.fact_sales_dashboard
WHERE order_date >= add_months(current_date(), -24)
""")

spark.sql(f"SELECT MIN(order_date), MAX(order_date), COUNT(*) FROM {GOLD}.v_fact_sales_incremental").show()

## Incremental refresh design

![Incremental refresh](../assets/images/incremental_refresh_range.png)

Power BI uses `RangeStart` and `RangeEnd` parameters. The recommended date
filter is half-open:

```sql
WHERE order_date >= RangeStart
  AND order_date <  RangeEnd
```

This avoids double-counting rows that fall exactly on a boundary.

In [ ]:
spark.sql(f"""
SELECT
  COUNT(*) AS rows_in_window,
  MIN(order_date) AS min_date,
  MAX(order_date) AS max_date
FROM {GOLD}.v_fact_sales_incremental
WHERE order_date >= DATE '2025-01-01'
  AND order_date <  DATE '2025-04-01'
""").show()

In [ ]:
spark.sql(f"""
EXPLAIN FORMATTED
SELECT order_date, category, customer_region, line_revenue
FROM {GOLD}.v_fact_sales_incremental
WHERE order_date >= DATE '2025-01-01'
  AND order_date <  DATE '2025-04-01'
""").show(80, truncate=False)

## BI contract

Use `docs/templates/bi-contract.md`.

Minimum decisions:

- Source for summary page: `gold.fact_sales_dashboard_monthly`.
- Source for detail/drill-through: `gold.v_fact_sales_incremental`.
- Baseline mode: Import.
- Live/DirectQuery only when freshness is more important than cost stability.

In [ ]:
spark.sql(f"""
SELECT
  year_month,
  customer_region,
  category,
  revenue,
  gross_margin,
  completed_orders,
  returned_orders
FROM {GOLD}.fact_sales_dashboard_monthly
ORDER BY year_month DESC, revenue DESC
LIMIT 20
""").show(truncate=False)

## Dataset table plan

| Power BI table | Source | Mode | Why |
|---|---|---|---|
| `Sales Monthly` | `gold.fact_sales_dashboard_monthly` | Import baseline | compact summary source |
| `Sales Detail` | `gold.v_fact_sales_incremental` | Import incremental or DirectQuery for live page | drill-through and freshness |
| `Date` | `gold.dim_date` | Import | stable dimension |
| `Product` | `gold.dim_product` | Import | slicers and category labels |
| `Customer` | `gold.dim_customer` | Import | segment and region slicers |

For a live demo, connect a small report page to the monthly aggregate first.
Avoid DirectQuery against line-grain Silver tables.

## Power BI thin layer example

Suggested Power BI measures:

```DAX
Revenue = SUM(fact_sales_dashboard_monthly[revenue])
Gross Margin % = DIVIDE(SUM(fact_sales_dashboard_monthly[gross_margin]), [Revenue])
Return Rate = DIVIDE(SUM(fact_sales_dashboard_monthly[returned_orders]), SUM(fact_sales_dashboard_monthly[completed_orders]) + SUM(fact_sales_dashboard_monthly[returned_orders]))
```

Everything else should be pushed into Databricks Gold unless there is a clear
reporting reason not to.

## Report layout walkthrough

Use `assets/images/powerbi_report_mockup.png` as the target report sketch:

- KPI cards at the top,
- revenue/margin trend as primary visual,
- region/category slices,
- filters that map to Gold columns,
- no hidden complex transformations in Power Query.

## Discussion: choose the mode

| Question | Import answer | DirectQuery answer |
|---|---|---|
| How fresh must the number be? | daily/hourly is enough | user needs current operational state |
| How many users click the report? | many readers | controlled audience |
| Where is cost paid? | scheduled refresh | every interaction |
| What Gold object is required? | curated table/view | aggregate or very selective view |